# EMG Accuracy Improvements - Google Colab GPU Training

This notebook runs the enhanced EMG classification with 10 strategic improvements:
1. Subject-wise normalization
2. Frequency-domain EMG features
3. AdamW + OneCycleLR scheduling
4. EMG augmentation (noise, scale, warp)
5. Conformer architecture
6. Dual-branch architecture (raw + features)
7. Supervised contrastive pre-training
8. CWT scalogram branch
9. Test-time augmentation (TTA)
10. Channel attention + label smoothing

**Setup Instructions:**
1. Connect to GPU runtime (Runtime → Change runtime type → T4 GPU)
2. Run cells in order
3. Results will be saved to Drive automatically

## 1. Check GPU and Environment

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('GPU Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    print('⚠️ WARNING: No GPU detected! Please change runtime to GPU.')

## 2. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ Google Drive mounted successfully')
except Exception as e:
    print(f'Drive mount skipped (running locally?): {e}')

## 3. Install Dependencies

In [ ]:
%%capture
# Install required packages quietly
%pip install -q numpy pandas scipy scikit-learn tqdm h5py PyWavelets
print('✓ Dependencies installed')
print('  - PyTorch (pre-installed in Colab)')
print('  - NumPy, Pandas, SciPy')
print('  - scikit-learn, tqdm, h5py')
print('  - PyWavelets (for CWT features)')

## 4. Configuration - Path Setup

In [ ]:
from pathlib import Path
import subprocess
import os

# ========== CONFIGURATION ==========

# Repository configuration
REPO_URL = 'https://github.com/MeghVyas3132/ULTRA-MoCap-Kinematics-Analysis.git'
REPO_NAME = 'ULTRA-MoCap-Kinematics-Analysis'

# Colab paths (auto-detected)
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    REPO_DIR = f'/content/{REPO_NAME}'
    CACHE_DIR = '/content/mocap_cache'
    
    # Clone repository if not exists
    if not Path(REPO_DIR).exists():
        print(f'📦 Cloning repository to {REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
        print('✓ Repository cloned')
    else:
        print(f'✓ Repository already exists at {REPO_DIR}')
    
    # Look for H5 dataset in Drive
    drive_paths = [
        '/content/drive/MyDrive/research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
        '/content/drive/MyDrive/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
        '/content/drive/MyDrive/research-paper/Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
    ]
    
    H5_PATH = None
    for p in drive_paths:
        if Path(p).exists():
            H5_PATH = p
            break
    
    if H5_PATH is None:
        print('❌ ERROR: All_subjects_data.h5 not found in expected Drive locations:')
        for p in drive_paths:
            print(f'  - {p}')
        print('\nPlease upload the dataset to one of these locations in your Google Drive.')
        raise FileNotFoundError('Dataset not found')
    
    DRIVE_RESULTS_DIR = '/content/drive/MyDrive/research-paper/results/EMG_improvements'
    
else:
    # Local paths
    REPO_DIR = '/Users/meghvyas/Desktop/research-paper'
    H5_PATH = '/Users/meghvyas/Desktop/research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5'
    CACHE_DIR = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/cache'
    DRIVE_RESULTS_DIR = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/EMG_improvements'

# Training configuration
DATASET_ROOT = f'{CACHE_DIR}/datasets'
RESULTS_FOLDER = f'{CACHE_DIR}/results/EMG_improvements'

# EMG Improvement Settings
EMG_MODEL_VARIANT = 'conformer'  # Options: 'conformer', 'dual_branch', 'lstm_msa', 'cwt_branch'
USE_AUGMENTATION = True
USE_TTA = True  # Test-time augmentation
USE_CONTRASTIVE_PRETRAIN = False  # Set True for contrastive pre-training (slower)
USE_LABEL_SMOOTHING = True
LABEL_SMOOTHING = 0.1

# Training hyperparameters
EMG_EPOCHS = 50
EMG_PATIENCE = 12
EMG_LR = 5e-4
EMG_BATCH_SIZE = 128

# LOSO configuration
MAX_FOLDS = 0  # 0 = all 13 subjects, or set 1-3 for quick testing
MODALITIES = 'emg'  # Focus on EMG only
RESULT_TAG = 'emg_improvements_v1'

# Create directories
Path(DATASET_ROOT).mkdir(parents=True, exist_ok=True)
Path(RESULTS_FOLDER).mkdir(parents=True, exist_ok=True)
Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# Print configuration
print('\n' + '='*60)
print('CONFIGURATION SUMMARY')
print('='*60)
print(f'Environment: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Repository: {REPO_DIR}')
print(f'Dataset: {H5_PATH}')
print(f'Cache: {CACHE_DIR}')
print(f'Results: {RESULTS_FOLDER}')
print(f'Drive Backup: {DRIVE_RESULTS_DIR}')
print(f'\nModel Variant: {EMG_MODEL_VARIANT}')
print(f'Augmentation: {USE_AUGMENTATION}')
print(f'TTA: {USE_TTA}')
print(f'Contrastive Pretrain: {USE_CONTRASTIVE_PRETRAIN}')
print(f'Label Smoothing: {LABEL_SMOOTHING if USE_LABEL_SMOOTHING else "Off"}')
print(f'\nEpochs: {EMG_EPOCHS}')
print(f'Learning Rate: {EMG_LR}')
print(f'Batch Size: {EMG_BATCH_SIZE}')
print(f'LOSO Folds: {"All 13" if MAX_FOLDS == 0 else MAX_FOLDS}')
print('='*60 + '\n')

## 5. Verify Files and Environment

In [ ]:
import sys

# Verify paths exist
assert Path(REPO_DIR).exists(), f'❌ Repository not found: {REPO_DIR}'
assert Path(H5_PATH).exists(), f'❌ Dataset not found: {H5_PATH}'

# Add repository to Python path
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB')

# Verify EMG improvements module exists
emg_improvements_path = Path(f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/scripts/training/emg_improvements.py')
assert emg_improvements_path.exists(), f'❌ EMG improvements module not found: {emg_improvements_path}'

print('✓ All paths verified')
print('✓ Python paths configured')
print('✓ EMG improvements module found')

# Try importing the improvements module
try:
    sys.path.insert(0, str(emg_improvements_path.parent))
    import emg_improvements
    print('✓ EMG improvements module imported successfully')
    print(f'  Available improvements: SubjectNormalizer, EMGConformer, DualBranchEMG, TTAWrapper, etc.')
except ImportError as e:
    print(f'⚠️ Warning: Could not import emg_improvements: {e}')
    print('  Will attempt to use module during training')

In [ ]:
# Debug: Check environment before training
import os
from pathlib import Path

print('='*60)
print('PRE-FLIGHT CHECKS')
print('='*60)

# 1. Check Python and PyTorch
import sys
import torch
print(f'\n1. Python: {sys.version.split()[0]}')
print(f'   PyTorch: {torch.__version__}')
print(f'   CUDA: {torch.cuda.is_available()}')

# 2. Check paths
print(f'\n2. Paths:')
print(f'   Repo exists: {Path(REPO_DIR).exists()}')
print(f'   Dataset exists: {Path(H5_PATH).exists()}')
if Path(H5_PATH).exists():
    size_gb = Path(H5_PATH).stat().st_size / (1024**3)
    print(f'   Dataset size: {size_gb:.2f} GB')

# 3. Check training script
print(f'\n3. Training scripts available:')
scripts_dir = Path(f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/scripts/training')
for script in sorted(scripts_dir.glob('*.py')):
    print(f'   - {script.name}')

# 4. Check emg_improvements module
print(f'\n4. EMG improvements module:')
emg_mod = Path(f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/scripts/training/emg_improvements.py')
print(f'   exists: {emg_mod.exists()}')
if emg_mod.exists():
    sys.path.insert(0, str(emg_mod.parent))
    try:
        import emg_improvements
        print(f'   ✓ Can import emg_improvements')
    except Exception as e:
        print(f'   ✗ Import failed: {e}')

# 5. Check dependencies
print(f'\n5. Dependencies:')
deps = ['numpy', 'pandas', 'scipy', 'sklearn', 'h5py', 'pywt']
for dep in deps:
    try:
        __import__(dep)
        print(f'   ✓ {dep}')
    except ImportError:
        print(f'   ✗ {dep} MISSING')

print(f'\n' + '='*60)
print('If all checks pass, proceed to next cell')
print('='*60)

## 6. Launch Training with EMG Improvements

In [ ]:
import subprocess
import sys
import os

# Verify GPU before training
assert torch.cuda.is_available(), '❌ GPU not available! Please change runtime to GPU.'
print(f'✓ Launching training on GPU: {torch.cuda.get_device_name(0)}\n')

# Set environment variables for training
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:512',
    
    # Data paths
    'DATA_H5_PATH': H5_PATH,
    'DATASET_ROOT': DATASET_ROOT,
    'RESULTS_FOLDER': RESULTS_FOLDER,
    'RESULT_TAG': RESULT_TAG,
    
    # Training config
    'MAX_FOLDS': str(MAX_FOLDS),
    'MODALITIES': MODALITIES,
    
    # EMG improvements
    'EMG_MODEL_VARIANT': EMG_MODEL_VARIANT,
    'EMG_EPOCHS': str(EMG_EPOCHS),
    'EMG_PATIENCE': str(EMG_PATIENCE),
    'EMG_LR': str(EMG_LR),
    'EMG_BATCH_SIZE': str(EMG_BATCH_SIZE),
    'USE_AUGMENTATION': str(int(USE_AUGMENTATION)),
    'USE_TTA': str(int(USE_TTA)),
    'USE_CONTRASTIVE_PRETRAIN': str(int(USE_CONTRASTIVE_PRETRAIN)),
    'USE_LABEL_SMOOTHING': str(int(USE_LABEL_SMOOTHING)),
    'LABEL_SMOOTHING': str(LABEL_SMOOTHING),
    
    # Performance optimizations
    'FAST_MODE': '1',
    'MATMUL_PRECISION': 'high',
    'DATALOADER_WORKERS': '4',
    'PREFETCH_FACTOR': '2',
    'PERSISTENT_WORKERS': '1',
})

# Try run_emg_colab.py first, then fall back to conv1d
scripts_to_try = [
    'run_emg_colab.py',
    'emg_optimized_loso.py',
    'conv1d_bigru_loso.py',
]

training_script = None
for script_name in scripts_to_try:
    script_path = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/scripts/training/{script_name}'
    if Path(script_path).exists():
        training_script = script_path
        print(f'✓ Found training script: {script_name}')
        break

if not training_script:
    print('❌ No training script found!')
    raise FileNotFoundError('Training script not found')

cmd = [sys.executable, '-u', training_script]

print(f'\n🚀 Starting EMG training with improvements...')
print(f'Script: {Path(training_script).name}')
print(f'Model: {EMG_MODEL_VARIANT}')
print(f'Folds: {"All 13" if MAX_FOLDS == 0 else MAX_FOLDS}')
print(f'Epochs: {EMG_EPOCHS}\n')
print('='*60)
print('Training output:')
print('='*60 + '\n')

# Run training - DO NOT capture output so errors are visible
try:
    result = subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print('\n' + '='*60)
    print('✅ Training completed successfully!')
    print('='*60)
except subprocess.CalledProcessError as e:
    print('\n' + '='*60)
    print(f'❌ Training failed with error code {e.returncode}')
    print('='*60)
    print('\nTroubleshooting:')
    print('1. Check error messages above')
    print('2. Verify dataset path is correct')
    print('3. Ensure all dependencies installed (Cell 3)')
    print('4. Try running debug cell (Cell 5.5) for details')
    raise
except KeyboardInterrupt:
    print('\n⚠️ Training interrupted by user')
    raise

## 7. Sync Results to Google Drive

In [ ]:
import shutil
from pathlib import Path
import datetime

print('📤 Syncing results to Google Drive...')

try:
    # Ensure Drive results directory exists
    Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    
    # Copy results folder
    if Path(RESULTS_FOLDER).exists():
        if Path(RESULTS_FOLDER).resolve() != Path(DRIVE_RESULTS_DIR).resolve():
            shutil.copytree(RESULTS_FOLDER, DRIVE_RESULTS_DIR, dirs_exist_ok=True)
            print(f'✓ Results synced to: {DRIVE_RESULTS_DIR}')
        else:
            print('ℹ️ Results already in Drive location')
        
        # Count and list result files
        result_files = list(Path(DRIVE_RESULTS_DIR).rglob('*.csv')) + \
                      list(Path(DRIVE_RESULTS_DIR).rglob('*.txt')) + \
                      list(Path(DRIVE_RESULTS_DIR).rglob('*.pth'))
        
        print(f'\n📊 Results Summary:')
        print(f'  Total files: {len(result_files)}')
        print(f'  Location: {DRIVE_RESULTS_DIR}')
        
        # Show key result files
        csv_files = [f for f in result_files if f.suffix == '.csv']
        if csv_files:
            print(f'\n  📈 Result CSVs:')
            for f in sorted(csv_files)[:10]:  # Show first 10
                print(f'    - {f.name}')
        
        model_files = [f for f in result_files if f.suffix == '.pth']
        if model_files:
            print(f'\n  🧠 Model Checkpoints:')
            for f in sorted(model_files)[:5]:  # Show first 5
                size_mb = f.stat().st_size / (1024*1024)
                print(f'    - {f.name} ({size_mb:.1f} MB)')
    else:
        print(f'⚠️ Results folder not found: {RESULTS_FOLDER}')
    
    # Create a timestamp file
    timestamp_file = Path(DRIVE_RESULTS_DIR) / 'last_sync.txt'
    with open(timestamp_file, 'w') as f:
        f.write(f'Last sync: {datetime.datetime.now()}\n')
        f.write(f'Model variant: {EMG_MODEL_VARIANT}\n')
        f.write(f'Epochs: {EMG_EPOCHS}\n')
        f.write(f'Folds: {"All 13" if MAX_FOLDS == 0 else MAX_FOLDS}\n')
    
    print(f'\n✅ Sync completed successfully!')
    print(f'\n💡 View your results in Google Drive:')
    print(f'   {DRIVE_RESULTS_DIR}')
    
except Exception as e:
    print(f'❌ Error during sync: {e}')
    print(f'Results may still be available locally at: {RESULTS_FOLDER}')

## 8. Quick Results Analysis

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print('📊 Analyzing Results...\n')

try:
    # Find the main results CSV
    result_csvs = list(Path(RESULTS_FOLDER).rglob('*summary*.csv')) + \
                  list(Path(RESULTS_FOLDER).rglob('*results*.csv'))
    
    if not result_csvs:
        print('⚠️ No results CSV files found yet.')
    else:
        # Load the most recent results
        latest_csv = sorted(result_csvs, key=lambda x: x.stat().st_mtime)[-1]
        print(f'Loading: {latest_csv.name}\n')
        
        df = pd.read_csv(latest_csv)
        
        # Display key metrics
        if 'accuracy' in df.columns or 'test_accuracy' in df.columns:
            acc_col = 'accuracy' if 'accuracy' in df.columns else 'test_accuracy'
            
            print('='*60)
            print('ACCURACY SUMMARY')
            print('='*60)
            print(f'Mean Accuracy: {df[acc_col].mean():.4f} ± {df[acc_col].std():.4f}')
            print(f'Best Fold: {df[acc_col].max():.4f}')
            print(f'Worst Fold: {df[acc_col].min():.4f}')
            print('='*60)
            
            # Per-fold breakdown
            if 'fold' in df.columns or 'subject' in df.columns:
                fold_col = 'fold' if 'fold' in df.columns else 'subject'
                print(f'\nPer-Fold Results:')
                for _, row in df.iterrows():
                    fold_id = row[fold_col]
                    acc = row[acc_col]
                    print(f'  {fold_col.capitalize()} {fold_id}: {acc:.4f}')
        else:
            print('Results columns:', df.columns.tolist())
            print('\nFirst few rows:')
            print(df.head())
        
        print(f'\n✓ Full results saved to: {latest_csv}')
        
except Exception as e:
    print(f'Could not analyze results: {e}')
    print(f'Results should be available at: {RESULTS_FOLDER}')

## 9. Compare with Baseline (Optional)

In [ ]:
print('📊 Comparison with Baseline\n')
print('='*60)
print('Model Configuration:')
print(f'  Variant: {EMG_MODEL_VARIANT}')
print(f'  Augmentation: {USE_AUGMENTATION}')
print(f'  TTA: {USE_TTA}')
print(f'  Contrastive Pretrain: {USE_CONTRASTIVE_PRETRAIN}')
print(f'  Label Smoothing: {LABEL_SMOOTHING if USE_LABEL_SMOOTHING else "Off"}')
print('='*60)
print('\nTo compare with baseline:')
print('1. Run this notebook with EMG_MODEL_VARIANT="lstm_msa" and improvements disabled')
print('2. Compare accuracy metrics between runs')
print('3. Best results are typically achieved with:')
print('   - EMG_MODEL_VARIANT="conformer" or "dual_branch"')
print('   - USE_AUGMENTATION=True')
print('   - USE_TTA=True')
print('   - USE_LABEL_SMOOTHING=True (0.1)')

## Notes

**Model Variants:**
- `conformer`: SOTA architecture combining convolution and attention
- `dual_branch`: Processes raw EMG + handcrafted features separately
- `lstm_msa`: LSTM with multi-scale attention (baseline)
- `cwt_branch`: Continuous wavelet transform scalograms

**Quick Testing:**
- Set `MAX_FOLDS=1` to test on just one subject (faster)
- Set `MAX_FOLDS=0` for full 13-fold LOSO validation

**Expected Improvements:**
- Baseline (LSTM): ~75-80% accuracy
- With improvements: ~82-87% accuracy (expected 5-10% gain)

**Troubleshooting:**
- If OOM errors occur: Reduce `EMG_BATCH_SIZE` to 64 or 32
- If training is slow: Ensure GPU runtime is selected
- If dataset not found: Check Google Drive paths in configuration cell